[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/100.%20The%20Closed-Loop%20%28Reverse%20Logistics%29%20Network%20Design%20Problem/P100-Tier-12_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 100. The Closed-Loop Network Design Problem
## Tier 12: Biologically Integrated and Regenerative Logistics

This notebook builds a small, illustrative logistics network whose nodes host modeled biological systems. It simulates operations and environmental feedback, records waste and regeneration measures, evaluates symbiotic relationships, and repeatedly simulates the network to track its best observed health score. The code is a simulation and iterative evaluation exercise; it does not choose node locations, routes, or network structure through a mathematical optimization model.

### Goals, inputs, and outputs

- **Goal:** Explore how simplified ecosystem-service, waste-processing, environmental-response, and relationship rules interact in a logistics-network simulation.
- **Inputs:** The specified node configurations, capacities, biological-system types, relationship benefits, random seed, and simulation settings.
- **Outputs:** Hourly and final network metrics, an iterative health evaluation, a descriptive report, and visual summaries.
- **Method:** Simulate network operations in time order; update biological-system health and relationships; then repeat one-hour simulations while recording the best observed network health.

### Scope and modeling assumptions

The model is an educational abstraction, not a calibrated ecological or logistics forecast. Capacities, health, biodiversity, and service values are dimensionless or illustrative unless a unit is explicitly stated. Waste generation is set to 10% of operating volume, and each biological system processes an equal share of a node's generated waste. A system's regeneration fraction is its health multiplied by its regenerative rate. Ecosystem-service scores reduce modeled waste generation and define a simple environmental-impact proxy; they are not measurements of real emissions or ecosystem outcomes. Carrying capacity is recorded for reporting but does not constrain operations or regeneration. The simulation does not model transport flows, costs, facility-location decisions, physical conservation of materials, or ecological time scales.

### Model equations

At node $i$, let $V_i$ be operating volume and let $S_i$ be the sum of the waste-reduction service scores supplied by its biological systems. The model limits the waste-reduction fraction to the interval $[0,1]$ so generated waste cannot become negative:

$$W_i = 0.1 V_i \left(1 - \min\left(1,\,0.5 S_i\right)\right).$$

If node $i$ has $m_i > 0$ biological systems, each receives an equal share of that waste. For system $j$, with health $h_{ij}$ and regenerative rate $r_{ij}$, the regenerated amount is

$$R_i = \sum_{j=1}^{m_i} \frac{W_i}{m_i} h_{ij} r_{ij}.$$

The node circularity score is $C_i = R_i/W_i$ when $W_i>0$, and is defined as $1$ when no waste is generated. With no biological systems, regeneration is zero. The network's hourly average circularity is the arithmetic mean of node scores. Its environmental-impact proxy is the sum of node proxies, where each node's proxy is $-0.1$ times the sum of its carbon-sequestration service scores; a negative value represents a modeled positive contribution under this convention.

For biological-system health, the environmental response is the mean of temperature, humidity, and pollution impacts, denoted $E$. Health is updated by

$$h^{\text{new}} = \operatorname{clip}\left(0.7h^{\text{old}} + 0.3E,\,0,\,1\right).$$

The network-health score combines average biological health $H$, average node circularity $C$, and average relationship strength $Q$:

$$N = 0.4H + 0.3C + 0.3Q.$$

The regenerative index is $I = HC$. These are constructed indicators, not empirically validated performance measures. The code implements the equations in `BiologicalSystem`, `RegenerativeNode`, and `RegenerativeLogisticsNetwork`.

### Iterative evaluation

The iterative procedure runs one-hour simulations and records the largest network-health score observed. If $N_k$ is network health after iteration $k$, the tracked value after $K$ iterations is

$$N^{\star}_K = \max\left(N_0, N_1, \ldots, N_K\right),$$

where $N_0$ is the health at the start of the procedure. The corresponding regenerative index is stored when a new best health is observed. This procedure does not change network structure, restore the network to its best historical state, or guarantee that later states improve. Its role is to evaluate the evolving simulation under repeated operations, not to solve a facility-location optimization problem. The computational work grows approximately linearly with the number of simulated hours, nodes, and biological systems; the relationship-processing work also grows with the number of relationships.

### What to examine

Interpret the reported scores as outputs of the stated rules. Compare hourly trends, node-level waste and regeneration, system-health summaries, and relationship metrics. A high circularity score here means that a large share of modeled generated waste is regenerated; it does not establish zero waste in a real supply chain. Recommendations are threshold-based prompts for discussion, not operational prescriptions.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field
from enum import Enum
import random
import time
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A consistent plotting style makes comparisons across the four panels easier.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Preserve the notebook's seeded random-number streams for repeatable simulations.
np.random.seed(42)
random.seed(42)

In [ ]:
class BiologicalSystemType(Enum):
    """Biological-system categories used by the illustrative network."""

    FOREST = "forest"
    WETLAND = "wetland"
    CORAL_REEF = "coral_reef"
    MICROBIOME = "microbiome"
    MYCELIAL_NETWORK = "mycelial_network"


@dataclass
class BiologicalSystem:
    """A simplified biological system attached to a logistics node."""

    system_id: str
    system_type: BiologicalSystemType
    location: Tuple[float, float]
    health: float = 1.0
    carrying_capacity: float = 1000.0
    regenerative_rate: float = 0.1
    biodiversity_index: float = 0.8
    ecosystem_services: List[str] = field(default_factory=list)

    def process_waste(self, waste_input: float) -> float:
        """Return the modeled resource amount regenerated from waste."""
        if waste_input < 0:
            raise ValueError("waste_input must be non-negative")

        if self.health <= 0.3:
            return 0.0

        # Health scales the system's regenerative rate, linking ecological
        # condition to the amount of waste that can be processed.
        processing_efficiency = self.health * self.regenerative_rate
        regenerated_resources = waste_input * processing_efficiency
        self.health = min(1.0, self.health + 0.01)
        return regenerated_resources

    def provide_ecosystem_service(self, service_type: str) -> float:
        """Return a dimensionless service score when the service is available."""
        if service_type not in self.ecosystem_services:
            return 0.0
        return self.health * self.biodiversity_index

    def evolve(self, environmental_conditions: Dict[str, float]) -> float:
        """Update health in response to modeled environmental conditions."""
        temperature = environmental_conditions.get("temperature", 20.0)
        humidity = environmental_conditions.get("humidity", 50.0)
        pollution = environmental_conditions.get("pollution", 0.0)

        if self.system_type == BiologicalSystemType.FOREST:
            seasonal_phase = environmental_conditions.get("seasonal_phase", 0.0)
            optimal_temp = 15.0 + 10.0 * np.sin(seasonal_phase)
            temp_impact = 1.0 - abs(temperature - optimal_temp) / 20.0
            humidity_impact = 1.0 - abs(humidity - 60.0) / 40.0
            pollution_impact = 1.0 - pollution
        elif self.system_type == BiologicalSystemType.WETLAND:
            optimal_temp = 20.0
            optimal_humidity = 80.0
            temp_impact = 1.0 - abs(temperature - optimal_temp) / 15.0
            humidity_impact = 1.0 - abs(humidity - optimal_humidity) / 30.0
            pollution_impact = 1.0 - pollution * 1.5
        else:
            temp_impact = 1.0 - abs(temperature - 20.0) / 20.0
            humidity_impact = 1.0 - abs(humidity - 50.0) / 50.0
            pollution_impact = 1.0 - pollution

        environmental_health = (
            temp_impact + humidity_impact + pollution_impact
        ) / 3.0
        self.health = float(
            np.clip(0.7 * self.health + 0.3 * environmental_health, 0.0, 1.0)
        )
        return self.health


@dataclass
class RegenerativeNode:
    """A logistics node with operational and biological attributes."""

    node_id: str
    location: Tuple[float, float]
    capacity: float
    biological_systems: List[BiologicalSystem] = field(default_factory=list)
    waste_generation: float = 0.0
    resource_regeneration: float = 0.0
    circularity_score: float = 0.0
    environmental_impact: float = 0.0

    def add_biological_system(self, system: BiologicalSystem) -> None:
        """Attach a biological system to this node."""
        self.biological_systems.append(system)

    def process_operations(self, operations_volume: float) -> Dict[str, float]:
        """Calculate waste, regeneration, circularity, and impact proxy."""
        if operations_volume < 0:
            raise ValueError("operations_volume must be non-negative")

        waste_reduction_score = sum(
            system.provide_ecosystem_service("waste_reduction")
            for system in self.biological_systems
        )
        waste_reduction_fraction = min(1.0, 0.5 * waste_reduction_score)
        self.waste_generation = (
            operations_volume * 0.1 * (1.0 - waste_reduction_fraction)
        )

        total_regeneration = 0.0
        if self.biological_systems:
            waste_share = self.waste_generation / len(self.biological_systems)
            for system in self.biological_systems:
                total_regeneration += system.process_waste(waste_share)

        self.resource_regeneration = total_regeneration
        if self.waste_generation > 0:
            self.circularity_score = total_regeneration / self.waste_generation
        else:
            self.circularity_score = 1.0

        carbon_service_score = sum(
            system.provide_ecosystem_service("carbon_sequestration")
            for system in self.biological_systems
        )
        self.environmental_impact = -0.1 * carbon_service_score

        return {
            "waste_generated": self.waste_generation,
            "resources_regenerated": total_regeneration,
            "circularity_score": self.circularity_score,
            "environmental_impact": self.environmental_impact,
        }


@dataclass
class SymbioticRelationship:
    """A simplified relationship and its two directional benefit scores."""

    relationship_id: str
    node_1_id: str
    node_2_id: str
    relationship_type: str
    benefit_1_to_2: float = 0.0
    benefit_2_to_1: float = 0.0
    strength: float = 1.0

    def calculate_mutual_benefit(self) -> float:
        """Combine directional benefits according to relationship type."""
        if self.relationship_type == "mutualism":
            return (self.benefit_1_to_2 + self.benefit_2_to_1) * self.strength
        if self.relationship_type == "commensalism":
            return max(self.benefit_1_to_2, self.benefit_2_to_1) * self.strength
        return (
            abs(self.benefit_1_to_2 - self.benefit_2_to_1)
            * self.strength
            * 0.5
        )


@dataclass
class RegenerativeLogisticsNetwork:
    """A network that simulates operations and biological feedback."""

    network_id: str
    nodes: Dict[str, RegenerativeNode] = field(default_factory=dict)
    relationships: List[SymbioticRelationship] = field(default_factory=list)
    environmental_conditions: Dict[str, float] = field(default_factory=dict)
    network_health: float = 1.0
    regenerative_index: float = 0.0

    def add_node(self, node: RegenerativeNode) -> None:
        """Add or replace a node by its identifier."""
        self.nodes[node.node_id] = node

    def add_relationship(self, relationship: SymbioticRelationship) -> None:
        """Record a relationship between network nodes."""
        self.relationships.append(relationship)

    def simulate_network_operations(
        self, duration_hours: int = 24
    ) -> Dict[str, Any]:
        """Simulate node operations, environmental response, and relationships."""
        if duration_hours < 0:
            raise ValueError("duration_hours must be non-negative")

        simulation_results = {
            "duration_hours": duration_hours,
            "hourly_metrics": [],
            "final_state": {},
            "network_evolution": [],
            "environmental_impact": 0.0,
        }

        for hour in range(duration_hours):
            self._update_environmental_conditions(hour)
            hourly_metrics = {
                "hour": hour + 1,
                "node_metrics": {},
                "network_metrics": {},
                "relationship_metrics": {},
            }

            total_waste = 0.0
            total_regeneration = 0.0
            total_circularity = 0.0
            total_environmental_impact = 0.0

            for node_id, node in self.nodes.items():
                for system in node.biological_systems:
                    system.evolve(self.environmental_conditions)

                operations_volume = node.capacity * np.random.uniform(0.5, 1.0)
                metrics = node.process_operations(operations_volume)
                total_waste += metrics["waste_generated"]
                total_regeneration += metrics["resources_regenerated"]
                total_circularity += metrics["circularity_score"]
                total_environmental_impact += metrics["environmental_impact"]
                hourly_metrics["node_metrics"][node_id] = metrics

            number_of_nodes = len(self.nodes)
            if number_of_nodes:
                total_capacity = sum(node.capacity for node in self.nodes.values())
                network_metrics = {
                    "total_waste": total_waste,
                    "total_regeneration": total_regeneration,
                    "average_circularity": total_circularity / number_of_nodes,
                    "total_environmental_impact": total_environmental_impact,
                    "waste_efficiency": 1.0 - total_waste / (total_capacity + 1e-6),
                }
            else:
                network_metrics = {}
            hourly_metrics["network_metrics"] = network_metrics

            total_mutual_benefit = 0.0
            for relationship in self.relationships:
                mutual_benefit = relationship.calculate_mutual_benefit()
                total_mutual_benefit += mutual_benefit
                if mutual_benefit > 0.5:
                    relationship.strength = min(1.0, relationship.strength + 0.01)
            hourly_metrics["relationship_metrics"][
                "total_mutual_benefit"
            ] = total_mutual_benefit

            simulation_results["hourly_metrics"].append(hourly_metrics)
            self._update_network_health()
            simulation_results["network_evolution"].append(
                {
                    "hour": hour + 1,
                    "network_health": self.network_health,
                    "regenerative_index": self.regenerative_index,
                }
            )

        hourly_network_metrics = [
            item["network_metrics"]
            for item in simulation_results["hourly_metrics"]
        ]
        simulation_results["final_state"] = {
            "network_health": self.network_health,
            "regenerative_index": self.regenerative_index,
            "total_environmental_impact": sum(
                metrics.get("total_environmental_impact", 0.0)
                for metrics in hourly_network_metrics
            ),
            "final_circularity": float(
                np.mean(
                    [
                        metrics.get("average_circularity", 0.0)
                        for metrics in hourly_network_metrics
                    ]
                )
            ) if hourly_network_metrics else 0.0,
        }
        return simulation_results

    def _update_environmental_conditions(self, hour: int) -> None:
        """Generate deterministic cycles plus seeded pollution and wind variation."""
        daily_phase = (hour % 24) / 24.0 * 2.0 * np.pi
        seasonal_phase = (hour / 24.0) / 365.0 * 2.0 * np.pi
        daily_temperature = 20.0 + 10.0 * np.sin(daily_phase - np.pi / 2.0)
        seasonal_temperature = 5.0 * np.sin(seasonal_phase)

        self.environmental_conditions = {
            "temperature": daily_temperature + seasonal_temperature,
            "humidity": 50.0 + 20.0 * np.sin(daily_phase),
            "pollution": max(
                0.0,
                0.1 * np.sin(hour / 12.0 * np.pi)
                + np.random.normal(0.0, 0.05),
            ),
            "sunlight": max(0.0, np.sin(daily_phase - np.pi / 2.0)),
            "wind_speed": 5.0 + 3.0 * np.random.random(),
            "seasonal_phase": seasonal_phase,
        }

    def _update_network_health(self) -> None:
        """Recompute the composite health and regenerative indicators."""
        if not self.nodes:
            self.network_health = 0.0
            self.regenerative_index = 0.0
            return

        all_systems = [
            system
            for node in self.nodes.values()
            for system in node.biological_systems
        ]
        average_system_health = (
            float(np.mean([system.health for system in all_systems]))
            if all_systems
            else 0.0
        )
        average_circularity = float(
            np.mean([node.circularity_score for node in self.nodes.values()])
        )
        average_relationship_strength = (
            float(np.mean([item.strength for item in self.relationships]))
            if self.relationships
            else 0.5
        )

        self.network_health = (
            0.4 * average_system_health
            + 0.3 * average_circularity
            + 0.3 * average_relationship_strength
        )
        self.regenerative_index = average_circularity * average_system_health


@dataclass
class BiologicalOptimizer:
    """Track the best observed health during repeated network simulations."""

    optimizer_id: str
    network: RegenerativeLogisticsNetwork
    optimization_history: List[Dict[str, Any]] = field(default_factory=list)

    def optimize_network_structure(
        self, max_iterations: int = 50
    ) -> Dict[str, Any]:
        """Iteratively simulate operations and retain the best health record."""
        if max_iterations < 0:
            raise ValueError("max_iterations must be non-negative")

        best_network_health = self.network.network_health
        best_regenerative_index = self.network.regenerative_index
        initial_health = best_network_health
        optimization_results = {
            "initial_health": initial_health,
            "initial_regenerative_index": best_regenerative_index,
            "final_health": best_network_health,
            "final_regenerative_index": best_regenerative_index,
            "improvement": 0.0,
            "iterations": max_iterations,
            "optimization_history": [],
        }

        for iteration in range(max_iterations):
            simulation = self.network.simulate_network_operations(duration_hours=1)
            current_health = self.network.network_health
            current_regenerative_index = self.network.regenerative_index

            if current_health > best_network_health:
                best_network_health = current_health
                best_regenerative_index = current_regenerative_index

            optimization_results["optimization_history"].append(
                {
                    "iteration": iteration + 1,
                    "network_health": current_health,
                    "regenerative_index": current_regenerative_index,
                    "environmental_impact": simulation["final_state"][
                        "total_environmental_impact"
                    ],
                }
            )

        optimization_results["final_health"] = best_network_health
        optimization_results["final_regenerative_index"] = (
            best_regenerative_index
        )
        if initial_health > 0:
            optimization_results["improvement"] = (
                best_network_health - initial_health
            ) / initial_health
        else:
            optimization_results["improvement"] = 0.0
        self.optimization_history.extend(
            optimization_results["optimization_history"]
        )
        return optimization_results

    def generate_biological_report(self) -> Dict[str, Any]:
        """Summarize the current network state and threshold-based prompts."""
        report = {
            "report_timestamp": time.time(),
            "network_id": self.network.network_id,
            "biological_integration": {
                "total_nodes": len(self.network.nodes),
                "total_biological_systems": sum(
                    len(node.biological_systems)
                    for node in self.network.nodes.values()
                ),
                "system_types": defaultdict(int),
                "average_health": 0.0,
                "total_carrying_capacity": 0.0,
            },
            "symbiotic_relationships": {
                "total_relationships": len(self.network.relationships),
                "relationship_types": defaultdict(int),
                "average_strength": 0.0,
                "total_mutual_benefit": 0.0,
            },
            "regenerative_performance": {
                "network_health": self.network.network_health,
                "regenerative_index": self.network.regenerative_index,
                "circularity_score": 0.0,
                "environmental_impact": 0.0,
            },
            "recommendations": [],
        }

        all_systems = [
            system
            for node in self.network.nodes.values()
            for system in node.biological_systems
        ]
        if all_systems:
            report["biological_integration"]["average_health"] = float(
                np.mean([system.health for system in all_systems])
            )
            report["biological_integration"]["total_carrying_capacity"] = sum(
                system.carrying_capacity for system in all_systems
            )
            for system in all_systems:
                report["biological_integration"]["system_types"][
                    system.system_type.value
                ] += 1

        if self.network.relationships:
            report["symbiotic_relationships"]["average_strength"] = float(
                np.mean([item.strength for item in self.network.relationships])
            )
            report["symbiotic_relationships"]["total_mutual_benefit"] = sum(
                item.calculate_mutual_benefit()
                for item in self.network.relationships
            )
            for relationship in self.network.relationships:
                report["symbiotic_relationships"]["relationship_types"][
                    relationship.relationship_type
                ] += 1

        if self.network.nodes:
            report["regenerative_performance"]["circularity_score"] = float(
                np.mean(
                    [node.circularity_score for node in self.network.nodes.values()]
                )
            )
            report["regenerative_performance"]["environmental_impact"] = sum(
                node.environmental_impact for node in self.network.nodes.values()
            )

        average_health = report["biological_integration"]["average_health"]
        circularity = report["regenerative_performance"]["circularity_score"]
        average_strength = report["symbiotic_relationships"]["average_strength"]
        environmental_impact = report["regenerative_performance"][
            "environmental_impact"
        ]

        if average_health < 0.7:
            report["recommendations"].append(
                "Review the modeled environmental conditions and biological-system health."
            )
        if circularity < 0.8:
            report["recommendations"].append(
                "Examine the modeled waste-processing and regeneration assumptions."
            )
        if average_strength < 0.6:
            report["recommendations"].append(
                "Review the benefit assumptions for relationships between nodes."
            )
        if environmental_impact > -10:
            report["recommendations"].append(
                "Review the ecosystem-service assumptions used by the impact proxy."
            )
        report["recommendations"].append(
            "Treat these indicators as model outputs and assess their assumptions before applying them."
        )
        return report

In [ ]:
# Build the example network from the stated node, system, and relationship data.
print("Initializing the biologically integrated logistics network...")
print("=" * 60)

network = RegenerativeLogisticsNetwork(network_id="bio_regenerative_001")

node_configurations = [
    ("forest_hub", (10, 10), 5000, [BiologicalSystemType.FOREST, BiologicalSystemType.MYCELIAL_NETWORK]),
    ("wetland_center", (30, 20), 3000, [BiologicalSystemType.WETLAND, BiologicalSystemType.MICROBIOME]),
    ("coral_facility", (50, 30), 2000, [BiologicalSystemType.CORAL_REEF]),
    ("mixed_ecology", (70, 15), 4000, [BiologicalSystemType.FOREST, BiologicalSystemType.WETLAND, BiologicalSystemType.MICROBIOME]),
    ("mycelial_network", (25, 45), 3500, [BiologicalSystemType.MYCELIAL_NETWORK, BiologicalSystemType.FOREST]),
]

print("Creating regenerative nodes...")
for node_id, location, capacity, system_types in node_configurations:
    node = RegenerativeNode(
        node_id=node_id,
        location=location,
        capacity=capacity,
    )

    for system_type in system_types:
        system = BiologicalSystem(
            system_id=f"{node_id}_{system_type.value}",
            system_type=system_type,
            location=(
                location[0] + np.random.uniform(-5, 5),
                location[1] + np.random.uniform(-5, 5),
            ),
            carrying_capacity=np.random.uniform(500, 1500),
            regenerative_rate=np.random.uniform(0.05, 0.2),
            biodiversity_index=np.random.uniform(0.6, 0.95),
        )

        if system_type == BiologicalSystemType.FOREST:
            system.ecosystem_services = [
                "carbon_sequestration",
                "waste_reduction",
                "biodiversity_support",
            ]
        elif system_type == BiologicalSystemType.WETLAND:
            system.ecosystem_services = [
                "water_purification",
                "flood_control",
                "waste_reduction",
            ]
        elif system_type == BiologicalSystemType.CORAL_REEF:
            system.ecosystem_services = [
                "biodiversity_support",
                "water_purification",
                "coastal_protection",
            ]
        elif system_type in (
            BiologicalSystemType.MICROBIOME,
            BiologicalSystemType.MYCELIAL_NETWORK,
        ):
            system.ecosystem_services = [
                "waste_reduction",
                "nutrient_cycling",
                "soil_enhancement",
            ]

        node.add_biological_system(system)

    network.add_node(node)
    print(
        f"  {node_id.replace('_', ' ').title()}: {capacity} capacity, "
        f"{len(node.biological_systems)} biological systems"
    )

print(f"\nTotal nodes created: {len(network.nodes)}")
print(
    "Total biological systems: "
    f"{sum(len(node.biological_systems) for node in network.nodes.values())}"
)

relationship_configurations = [
    ("forest_hub", "wetland_center", "mutualism", 0.8, 0.7),
    ("forest_hub", "mycelial_network", "mutualism", 0.9, 0.8),
    ("wetland_center", "coral_facility", "commensalism", 0.6, 0.4),
    ("mixed_ecology", "forest_hub", "mutualism", 0.7, 0.6),
    ("mixed_ecology", "mycelial_network", "mutualism", 0.8, 0.7),
]

print("\nCreating symbiotic relationships...")
for node1_id, node2_id, relationship_type, benefit_1_to_2, benefit_2_to_1 in relationship_configurations:
    relationship = SymbioticRelationship(
        relationship_id=f"{node1_id}_{node2_id}",
        node_1_id=node1_id,
        node_2_id=node2_id,
        relationship_type=relationship_type,
        benefit_1_to_2=benefit_1_to_2,
        benefit_2_to_1=benefit_2_to_1,
        strength=np.random.uniform(0.7, 1.0),
    )
    network.add_relationship(relationship)
    print(
        f"  {node1_id.replace('_', ' ').title()} ↔ "
        f"{node2_id.replace('_', ' ').title()}: "
        f"{relationship_type.title()} (modeled benefit: "
        f"{relationship.calculate_mutual_benefit():.2f})"
    )

print(f"\nTotal relationships created: {len(network.relationships)}")

optimizer = BiologicalOptimizer(
    optimizer_id="bio_optimizer_001",
    network=network,
)
print(f"\nEvaluation procedure initialized: {optimizer.optimizer_id}")
print(f"Network ID: {network.network_id}")
print(f"Initial network health: {network.network_health:.3f}")
print(f"Initial regenerative index: {network.regenerative_index:.3f}")

In [ ]:
# First simulate the stated 24-hour operating period, then evaluate repeated
# one-hour simulations while tracking the best observed health score.
print("Running the 24-hour biological-network simulation...")
simulation_results = network.simulate_network_operations(duration_hours=24)

final_state = simulation_results["final_state"]
print("\nSimulation summary:")
print(f"Duration: {simulation_results['duration_hours']} hours")
print(f"Final network health: {final_state['network_health']:.3f}")
print(f"Final regenerative index: {final_state['regenerative_index']:.3f}")
print(f"Mean hourly circularity: {final_state['final_circularity']:.3f}")
print(
    "Cumulative modeled environmental-impact proxy: "
    f"{final_state['total_environmental_impact']:.2f}"
)

if simulation_results["hourly_metrics"]:
    final_metrics = simulation_results["hourly_metrics"][-1]["network_metrics"]
    relationship_metrics = simulation_results["hourly_metrics"][-1][
        "relationship_metrics"
    ]
    print("\nLast-hour network metrics:")
    print(f"  Total waste generated: {final_metrics.get('total_waste', 0):.2f}")
    print(
        "  Total resources regenerated: "
        f"{final_metrics.get('total_regeneration', 0):.2f}"
    )
    print(
        "  Average circularity: "
        f"{final_metrics.get('average_circularity', 0):.3f}"
    )
    print(
        "  Waste efficiency indicator: "
        f"{final_metrics.get('waste_efficiency', 0):.3f}"
    )
    print(
        "  Total relationship benefit: "
        f"{relationship_metrics.get('total_mutual_benefit', 0):.2f}"
    )

print("\nEvaluating repeated network simulations...")
optimization_results = optimizer.optimize_network_structure(max_iterations=20)
print("\nBest-observed health record:")
print(f"Initial network health: {optimization_results['initial_health']:.3f}")
print(f"Best observed network health: {optimization_results['final_health']:.3f}")
print(
    "Regenerative index at that health record: "
    f"{optimization_results['final_regenerative_index']:.3f}"
)
print(f"Relative health change: {optimization_results['improvement']:.1%}")
print(f"One-hour simulation iterations: {optimization_results['iterations']}")

biological_report = optimizer.generate_biological_report()
bio_integration = biological_report["biological_integration"]
symbiotic = biological_report["symbiotic_relationships"]
regenerative = biological_report["regenerative_performance"]

print("\nBiological integration report:")
print(f"  Nodes: {bio_integration['total_nodes']}")
print(f"  Biological systems: {bio_integration['total_biological_systems']}")
print(f"  Average system health: {bio_integration['average_health']:.3f}")
print(
    "  Recorded carrying capacity: "
    f"{bio_integration['total_carrying_capacity']:.0f}"
)
print("  System types:")
for system_type, count in bio_integration["system_types"].items():
    print(f"    {system_type.replace('_', ' ').title()}: {count}")

print("\nSymbiotic relationships:")
print(f"  Relationships: {symbiotic['total_relationships']}")
print(f"  Average strength: {symbiotic['average_strength']:.3f}")
print(f"  Total modeled benefit: {symbiotic['total_mutual_benefit']:.2f}")
print("  Relationship types:")
for relationship_type, count in symbiotic["relationship_types"].items():
    print(f"    {relationship_type.title()}: {count}")

print("\nRegenerative performance indicators:")
print(f"  Network health: {regenerative['network_health']:.3f}")
print(f"  Regenerative index: {regenerative['regenerative_index']:.3f}")
print(f"  Current average node circularity: {regenerative['circularity_score']:.3f}")
print(
    "  Current environmental-impact proxy: "
    f"{regenerative['environmental_impact']:.2f}"
)
print("\nModel-based review prompts:")
for number, recommendation in enumerate(biological_report["recommendations"], 1):
    print(f"  {number}. {recommendation}")

# These checks connect reported aggregates to the hourly and node-level records.
assert len(simulation_results["hourly_metrics"]) == 24
assert len(simulation_results["network_evolution"]) == 24
assert all(node.waste_generation >= 0 for node in network.nodes.values())
assert all(node.resource_regeneration >= 0 for node in network.nodes.values())

In [ ]:
def visualize_biological_integration(
    network: RegenerativeLogisticsNetwork,
    simulation_results: Dict[str, Any],
    optimization_results: Dict[str, Any],
) -> Dict[str, Any]:
    """Plot simulation indicators and print descriptive summaries."""
    figure, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(16, 12)
    )

    evolution = simulation_results["network_evolution"]
    hours = [entry["hour"] for entry in evolution]
    health_scores = [entry["network_health"] for entry in evolution]
    regenerative_indices = [entry["regenerative_index"] for entry in evolution]

    ax1.set_title("Network Health and Regenerative Index")
    ax1.plot(hours, health_scores, marker="o", label="Network health")
    ax1.plot(
        hours,
        regenerative_indices,
        marker="s",
        label="Regenerative index",
    )
    ax1.set_xlabel("Simulation hour")
    ax1.set_ylabel("Dimensionless score")
    ax1.set_ylim(0, 1)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    hourly_metrics = simulation_results["hourly_metrics"]
    hourly_circularity = [
        item["network_metrics"].get("average_circularity", 0.0)
        for item in hourly_metrics
    ]
    hourly_environmental_impact = [
        item["network_metrics"].get("total_environmental_impact", 0.0)
        for item in hourly_metrics
    ]

    ax2.set_title("Circularity and Environmental-Impact Proxy")
    circularity_line, = ax2.plot(
        hours,
        hourly_circularity,
        color="tab:blue",
        label="Average circularity",
    )
    ax2.set_xlabel("Simulation hour")
    ax2.set_ylabel("Average circularity (dimensionless)", color="tab:blue")
    ax2.tick_params(axis="y", labelcolor="tab:blue")
    impact_axis = ax2.twinx()
    impact_line, = impact_axis.plot(
        hours,
        hourly_environmental_impact,
        color="tab:red",
        label="Impact proxy",
    )
    impact_axis.set_ylabel("Impact proxy (modeled units)", color="tab:red")
    impact_axis.tick_params(axis="y", labelcolor="tab:red")
    ax2.legend(
        [circularity_line, impact_line],
        [circularity_line.get_label(), impact_line.get_label()],
        loc="best",
    )
    ax2.grid(True, alpha=0.3)

    system_distribution = defaultdict(int)
    for node in network.nodes.values():
        for system in node.biological_systems:
            system_distribution[system.system_type.value] += 1

    ax3.set_title("Biological Systems by Type")
    if system_distribution:
        system_names = list(system_distribution)
        system_counts = [system_distribution[name] for name in system_names]
        bars = ax3.bar(system_names, system_counts, alpha=0.8)
        positions = np.arange(len(system_names))
        ax3.set_xticks(positions)
        ax3.set_xticklabels(
            [name.replace("_", " ").title() for name in system_names],
            rotation=35,
            ha="right",
        )
        ax3.set_ylabel("Number of systems")
        ax3.grid(True, axis="y", alpha=0.3)
        for bar, count in zip(bars, system_counts):
            ax3.text(
                bar.get_x() + bar.get_width() / 2.0,
                bar.get_height(),
                str(count),
                ha="center",
                va="bottom",
            )
    else:
        ax3.text(0.5, 0.5, "No biological systems", ha="center", va="center")
        ax3.set_axis_off()

    ax4.set_title("Relationship Benefits and Strengths")
    if network.relationships:
        relationship_names = [
            f"{item.node_1_id} ↔ {item.node_2_id}"
            for item in network.relationships
        ]
        benefits = [
            item.calculate_mutual_benefit() for item in network.relationships
        ]
        strengths = [item.strength for item in network.relationships]
        positions = np.arange(len(relationship_names))
        bar_width = 0.35
        ax4.bar(
            positions - bar_width / 2.0,
            benefits,
            bar_width,
            label="Modeled benefit",
            alpha=0.8,
        )
        ax4.bar(
            positions + bar_width / 2.0,
            strengths,
            bar_width,
            label="Relationship strength",
            alpha=0.8,
        )
        ax4.set_xlabel("Node pair")
        ax4.set_ylabel("Dimensionless value")
        ax4.set_xticks(positions)
        ax4.set_xticklabels(relationship_names, rotation=35, ha="right")
        ax4.legend()
        ax4.grid(True, axis="y", alpha=0.3)
    else:
        ax4.text(0.5, 0.5, "No relationships", ha="center", va="center")
        ax4.set_axis_off()

    figure.tight_layout()
    plt.show()

    all_systems = [
        system
        for node in network.nodes.values()
        for system in node.biological_systems
    ]
    print("\nBiological integration summary:")
    if all_systems:
        average_health = float(np.mean([system.health for system in all_systems]))
        print(f"Average biological-system health: {average_health:.3f}")
        health_by_type = defaultdict(list)
        for system in all_systems:
            health_by_type[system.system_type.value].append(system.health)
        print("Health by system type (mean and standard deviation):")
        for system_type, health_values in health_by_type.items():
            print(
                f"  {system_type.replace('_', ' ').title()}: "
                f"{np.mean(health_values):.3f} "
                f"(SD: {np.std(health_values):.3f})"
            )

    final_state = simulation_results["final_state"]
    final_circularity = final_state["final_circularity"]
    print("\nCircularity interpretation:")
    print(f"Mean hourly circularity: {final_circularity:.3f}")
    print(
        "This is the modeled regenerated-waste fraction, not a measured "
        "real-world circularity rate."
    )

    total_environmental_impact = final_state["total_environmental_impact"]
    print("\nEnvironmental-impact proxy interpretation:")
    print(f"Cumulative proxy: {total_environmental_impact:.2f}")
    print("Negative values denote modeled positive contributions by convention.")

    if network.relationships:
        average_benefit = float(
            np.mean(
                [item.calculate_mutual_benefit() for item in network.relationships]
            )
        )
        average_strength = float(
            np.mean([item.strength for item in network.relationships])
        )
        print("\nRelationship indicators:")
        print(f"Average modeled benefit: {average_benefit:.3f}")
        print(f"Average relationship strength: {average_strength:.3f}")

    print("\nIterative evaluation record:")
    print(f"Best observed network health: {optimization_results['final_health']:.3f}")
    print(
        "Regenerative index paired with that record: "
        f"{optimization_results['final_regenerative_index']:.3f}"
    )
    print(
        "The procedure records a best observed score; it does not alter "
        "network structure or restore a best historical state."
    )
    return simulation_results


visualized_results = visualize_biological_integration(
    network,
    simulation_results,
    optimization_results,
)

## Interpreting the model

The simulation combines node-level operations with simple rules for biological health, waste processing, and relationships. Each plotted series is a consequence of those rules and the specified example data; none should be interpreted as a forecast without empirical calibration and validation.

### Strengths and limitations of this abstraction

**What the model makes visible**

- It links biological-system health to modeled waste processing and ecosystem-service scores.
- It tracks circularity and a signed environmental-impact proxy over simulated hours.
- It represents relationship benefits and lets sufficiently beneficial relationships strengthen up to the model's upper bound.
- Its explicit rules make it possible to inspect how assumptions contribute to reported indicators.

**What the model does not establish**

- It does not demonstrate zero-waste operations, a positive real-world environmental impact, or self-organizing improvement.
- It does not optimize facility locations, transportation links, capacity allocation, or network topology.
- It omits costs, transport flows, material balances, biological carrying-capacity constraints, and empirically estimated ecological dynamics.
- Its thresholds and recommendations are illustrative decision prompts rather than validated standards.

### When this kind of model can be useful

A small simulation can help explain feedback mechanisms and identify which assumptions drive a proposed indicator. Before using the results for sustainability claims or operational decisions, define measurable units, calibrate parameters with relevant data, validate the model against observations, and include the physical and economic constraints required by the application. Extending the example with additional data or constraints would change its scope; the present notebook focuses on the stated simulation and iterative evaluation procedure.